In [29]:
# -------------------------------------------------
# Cell 1 – Import, configurazione e parametri
# -------------------------------------------------
import os, io, time
import threading
import collections, struct
import pyaudio, wave
from huggingface_hub import InferenceClient

# ---------- Parametri audio ----------
FORMAT = pyaudio.paInt16          # 16‑bit PCM
CHANNELS = 1                      # microfono mono
RATE = 16000                      # 16 kHz (standard per Whisper)
CHUNK = 1024                      # dimensione del buffer audio

# ---------- Parametri real‑time (semplificati) ----------
CHUNK_SECONDS = 5                 # durata (in secondi) di ogni blocco da trascrivere
CHUNK_FRAMES = int(RATE / CHUNK * CHUNK_SECONDS)   # n° di chunk da accumulare

# ---------- Parole‑trigger ----------
STOP_WORDS = {"stop", "ferma", "interrompi", "basta"}

# ---------- Token HuggingFace (facoltativo) ----------
HF_TOKEN_READ = os.getenv("HF_TOKEN_READ")   # se non è già impostato nel tuo ambiente
client = InferenceClient(
    provider="auto",
    token=HF_TOKEN_READ,
)


In [30]:
# -------------------------------------------------
# Cell 2 – Trascrizione, trigger e thread di registrazione
# -------------------------------------------------

# ------------------------------------------------------------------
# Stato globale condiviso tra le funzioni
# ------------------------------------------------------------------
audio_state = {
    "is_recording": False,      # True se la registrazione è attiva
    "stop_requested": False,    # True se è stato richiesto (manuale o vocale) di fermarsi
    "p": pyaudio.PyAudio(),    # istanza PyAudio
    "stream": None,            # stream del microfono
    "thread": None,            # thread di registrazione
    "frames": collections.deque(maxlen=CHUNK_FRAMES)  # accumula i chunk
}

# ------------------------------------------------------------------
# Funzione di trascrizione di un singolo blocco WAV in memoria
# ------------------------------------------------------------------
def _transcribe_chunk(raw_audio_bytes: bytes) -> str:
    """Converte `raw_audio_bytes` in WAV, lo invia a Whisper e restituisce il testo."""
    # 1️⃣ Costruzione WAV in RAM
    wav_buf = io.BytesIO()
    with wave.open(wav_buf, "wb") as wf:
        wf.setnchannels(CHANNELS)
        wf.setsampwidth(audio_state["p"].get_sample_size(FORMAT))
        wf.setframerate(RATE)
        wf.writeframes(raw_audio_bytes)
    wav_buf.seek(0)

    # 2️⃣ Chiamata API
    try:
        result = client.automatic_speech_recognition(
            model="openai/whisper-large-v3-turbo",
            audio=wav_buf.getvalue()
        )
        txt = result.get("text", "") if isinstance(result, dict) else str(result)
        return txt.strip()
    except Exception as e:
        print(f"❌ Errore trascrizione: {e}")
        return ""

# ------------------------------------------------------------------
# Controllo rapido delle parole‑trigger
# ------------------------------------------------------------------
def _contains_stop_word(text: str) -> bool:
    """Ritorna True se `text` (case‑insensitive) contiene una delle parole STOP_WORDS."""
    low = text.lower()
    return any(w in low for w in STOP_WORDS)


# ------------------------------------------------------------------
# Funzione di filtro per evitare di fare chiamate all'Api durante fasi di silenzio
# ------------------------------------------------------------------
def _rms(data_bytes: bytes) -> float:
    count = len(data_bytes) // 2
    samples = struct.unpack("<" + "h" * count, data_bytes)
    return (sum(s*s for s in samples) / max(count, 1)) ** 0.5

# ------------------------------------------------------------------
# Thread di registrazione (lavoro vero e proprio)
# ------------------------------------------------------------------
def _record_loop():
    """
    Legge dal microfono e accumula i chunk in `audio_state["frames"]`.
    Quando è stato raccolto un numero sufficiente di frame (CHUNK_FRAMES),
    trascrive quel blocco, lo stampa e controlla le parole‑trigger.
    """
    p = audio_state["p"]
    try:
        stream = p.open(
            format=FORMAT,
            channels=CHANNELS,
            rate=RATE,
            input=True,
            frames_per_buffer=CHUNK,
        )
        audio_state["stream"] = stream
    except Exception as e:
        print(f"❌ Impossibile aprire lo stream audio: {e}")
        print("Verifica che il microfono sia collegato.")
        audio_state["is_recording"] = False
        return

    frames = audio_state["frames"]  # alias per sintassi più chiara
    print("🟢 Registrazione in corso – blocchi di", CHUNK_SECONDS, "s")

    while audio_state["is_recording"]:
        # 1️⃣ Leggi un chunk dal microfono
        try:
            data = stream.read(CHUNK, exception_on_overflow=False)

            frames.append(data)
            # # Solo i blocchi con un minimo di energia vengono aggiunti
            # if _rms(data) > 300.0:  # soglia da regolare in base al rumore
            #     frames.append(data)
        except Exception as e:
            print(f"⚠️ Errore lettura audio: {e}")
            break

        

        # 2️⃣ Quando abbiamo raccolto abbastanza frame, trascrivi
        if len(frames) >= CHUNK_FRAMES:
            # Costruisci il blocco grezzo (WAV) e rimuovi i frame usati
            chunk_bytes = b"".join(list(frames)[:CHUNK_FRAMES])
            for _ in range(CHUNK_FRAMES):
                frames.popleft()

            # 3️⃣ Trascrizione
            transcript = _transcribe_chunk(chunk_bytes)
            if transcript:
                print("▶️", transcript)

                # 4️⃣ Controllo parole‑trigger (parlato)
                if _contains_stop_word(transcript):
                    audio_state["stop_requested"] = True
                    break

        # 5️⃣ Se è stato richiesto di fermare (manuale o vocale), esci dal ciclo
        if audio_state["stop_requested"]:
            break

    # Pulizia stream
    if stream:
        stream.stop_stream()
        stream.close()
    audio_state["stream"] = None
    print("🛑 Registrazione interrotta.")


# ------------------------------------------------------------------
# Funzioni di avvio / stop per il notebook 
# ------------------------------------------------------------------
def start_recording():
    """Avvia la registrazione (thread separato)."""
    if audio_state["is_recording"]:
        print("⏩ La registrazione è già in corso.")
        return
    audio_state["is_recording"] = True
    audio_state["stop_requested"] = False
    t = threading.Thread(target=_record_loop, daemon=True)
    audio_state["thread"] = t
    t.start()
    print("🔴 Registrazione avviata – pronuncia una delle parole‑trigger per fermarla.")

def stop_recording():
    """Ferma la registrazione (manuale)."""
    if not audio_state["is_recording"]:
        print("⏸️ Nessuna registrazione attiva.")
        return
    audio_state["stop_requested"] = True   # segnala al thread di fermarsi
    audio_state["is_recording"] = False
    if audio_state["thread"]:
        audio_state["thread"].join()
    audio_state["thread"] = None
    print("⏹️ Registrazione terminata.")

print("✅ Sistema di registrazione/trascrizione semplificato definito.")


✅ Sistema di registrazione/trascrizione semplificato definito.


In [31]:
# -------------------------------------------------
# Cell 3 – Loop di controllo interattivo
# -------------------------------------------------
def main():
    print("\n🎤 SISTEMA DI TRASCRIZIONE AUDIO REAL‑TIME")
    print("=" * 60)
    print("Comandi disponibili:")
    print("  start   → avvia la registrazione (blocchi di {} s)".format(CHUNK_SECONDS))
    print("  stop    → ferma la registrazione manualmente")
    print("  exit    → chiude il programma")
    print("\nParole‑trigger vocali: {}".format(", ".join(sorted(STOP_WORDS))))
    print("=" * 60)

    while True:
        try:
            cmd = input("\n🔹 Comando (start/stop/exit): ").strip().lower()
        except (EOFError, KeyboardInterrupt):
            print("\n👋 Uscita dal programma.")
            break

        if cmd == "start":
            start_recording()
        elif cmd == "stop":
            print("⏹️ Fermando la registrazione...")
            stop_recording()
        elif cmd == "exit":
            if audio_state["is_recording"]:
                print("⚠️ Registrazione ancora attiva. Ferma prima di uscire.")
                continue
            print("👋 Arrivederci!")
            break
        else:
            print("❓ Comando non riconosciuto. Usa: start, stop, exit")

print("✅ Loop di controllo definito.")


✅ Loop di controllo definito.


In [32]:
# -------------------------------------------------
# Cell 4 – Avvio del sistema di trascrizione
# -------------------------------------------------
# Esegui questa cella per iniziare a usare il notebook
main()



🎤 SISTEMA DI TRASCRIZIONE AUDIO REAL‑TIME
Comandi disponibili:
  start   → avvia la registrazione (blocchi di 5 s)
  stop    → ferma la registrazione manualmente
  exit    → chiude il programma

Parole‑trigger vocali: basta, ferma, interrompi, stop



🔹 Comando (start/stop/exit):  start


🔴 Registrazione avviata – pronuncia una delle parole‑trigger per fermarla.
🟢 Registrazione in corso – blocchi di 5 s
❌ Errore trascrizione: (Request ID: Root=1-691cc348-461b818822806a2529058df7;4a3dbea5-2fa3-4685-90f6-8a3c1bd01a4d)

Bad request:
Content type "None" not supported.
                Supported content types are:
                application/json, application/json; charset=UTF-8, text/csv, text/plain, image/png, image/jpeg, image/jpg, image/tiff, image/bmp, image/gif, image/webp, image/x-image, audio/x-flac, audio/flac, audio/mpeg, audio/x-mpeg-3, audio/wave, audio/wav, audio/x-wav, audio/ogg, audio/x-audio, audio/webm, audio/webm;codecs=opus, audio/AMR, audio/amr, audio/AMR-WB, audio/AMR-WB+, audio/m4a, audio/x-m4a
❌ Errore trascrizione: (Request ID: Root=1-691cc34d-755b0d80634286c279a191ee;2eb95463-3efe-4776-b3f4-f3ba2a76ed26)

Bad request:
Content type "None" not supported.
                Supported content types are:
                application/json, application/json; cha


🔹 Comando (start/stop/exit):  stop


⏹️ Fermando la registrazione...
🛑 Registrazione interrotta.
⏹️ Registrazione terminata.



🔹 Comando (start/stop/exit):  exit


👋 Arrivederci!
